# Day 12 · Zillacode prep 1/4: columns, filters, conditions, nulls and text

The theory days taught how Spark runs; the Zoomcamp module showed Spark on real data. Days 12 to 15 teach what Zillacode's 52 PySpark problems need on top of that: the **DataFrame API itself**, function by function, with the traps that make an answer fail. Each lesson explains one tool in depth with examples you run, followed by a few quick exercises.

Today:
0. How a Zillacode problem is run and checked
1. Columns: `select`, `withColumn`, expressions and types
2. Filtering rows
3. Conditional logic: `when` / `otherwise`
4. Missing values (null)
5. Text functions
6. Regular expressions
7. Duplicates, sorting and unions

In Zillacode's topic list this is *Conditional Logic*, *Case When Statements*, *Null Handling*, *String Manipulation*, *Regular Expressions*, *Distinct Function* and *Union Statements*.

## How to work
* Run the **Setup** cell first, then the cell that prints the tables.
* **Lessons** explain one tool at a time. Read them, run every example cell (the output is already shown under it), then change the example and run it again until you can predict the result.
* **Exercises** come right after each lesson and are in Zillacode's format: complete `etl(...)` in the cell under the task so that it **returns** a DataFrame, then run the cell. The last line, `etl(...).show()`, prints your result. It must match the expected output: same columns, same rows, same order. (Zillacode itself ignores the order of rows and columns; the expected outputs here are sorted only so you can compare them by eye.)
* The answers are collapsed at the very end of the notebook. Open one only after you have tried, then close it and write the solution again yourself.
* Today: about 1 hour of lessons and 2 hours of exercises (20 quick ones, then 3 mixed problems).

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 jupyterlab`, then `jupyter lab` and open this notebook.

## The data
A small online shop. The setup cell creates every table from rows you can read, and the cell after it prints them.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)
* **reviews**: *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating* (1 to 5), *comment* (messy text), *tags* (text like `fast delivery;good price`)
* **signups_raw**: sign-ups exactly as a web form sent them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text in three formats), *amount* (text), *prefs* (JSON text)
* **customers_new**: the customers table one week later
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*
* **referrals**: *referrer_id* (the customer who invited someone), *referred_id* (the customer who joined), *referral_date*

The data has the traps that Zillacode's hidden tests like to use: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is not in *products*; review 6 has no tags; two events were loaded twice; referral 5 points to customer 9, who doesn't exist.

## Setup
Run this cell once. It starts Spark and creates the tables.

In [2]:
# Setup: starts Spark and creates the tables. Run it once.
import logging, os, sys
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F, Window as W   # the same names as in Zillacode's editor

spark = (SparkSession.builder
         .master("local[4]")
         .appName("zillacode-prep-12")
         .config("spark.sql.ansi.enabled", "false")        # behave like Zillacode (Spark 3): bad text gives null, not an error
         .config("spark.sql.shuffle.partitions", "4")      # small data: 4 shuffle partitions are enough
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")               # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers = spark.createDataFrame([
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
], "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE")

products = spark.createDataFrame([
    # product_id, product_name, category, price (dollars for one item)
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE")

orders = spark.createDataFrame([
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING")

reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags (text)
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ",    "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                     "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow",  "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",          "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",               "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",                None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",                  "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",         "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text, three formats), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

customers_new = spark.createDataFrame([
    # the customers table one week later: 2 moved, 6 has an email now, 7 is gone, 8 is new
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date (both days included), discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

events = spark.createDataFrame([
    # customer_id, event_time, page (two rows were loaded twice)
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

referrals = spark.createDataFrame([
    # referrer_id (the customer who invited), referred_id (the customer who joined), referral_date
    (1, 4, date(2024, 2, 10)),
    (4, 5, date(2024, 2, 25)),
    (1, 6, date(2024, 3, 8)),
    (2, 7, date(2024, 3, 12)),
    (2, 9, date(2024, 3, 30)),
], "referrer_id INT, referred_id INT, referral_date DATE")

print("Spark", spark.version, "is ready. Tables: customers, products, orders, reviews, signups_raw, customers_new, "
      "promotions, events, referrals")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 11:18:53 WARN Utils: Your hostname, Usmanovs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.98.226.138 instead (on interface en0)
26/11/11 11:18:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 11:18:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "FATAL".


Spark 4.2.0 is ready. Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals


Print every table, so you can check results by eye:

In [3]:
for name in ["customers", "products", "orders", "reviews", "signups_raw", "customers_new", "promotions", "events", "referrals"]:
    print(name)
    globals()[name].show(truncate=False)

customers
+-----------+----------------+----------------+--------+-----------+
|customer_id|name            |email           |city    |signup_date|
+-----------+----------------+----------------+--------+-----------+
|1          |Aziz Karimov    |aziz@mail.uz    |Tashkent|2024-01-05 |
|2          |Bella Schmidt   |bella@web.de    |Berlin  |2024-01-20 |
|3          |Carlos Ruiz     |carlos@correo.es|Madrid  |2024-02-02 |
|4          |Dilnoza Rahimova|dilnoza@mail.uz |Tashkent|2024-02-14 |
|5          |Emma Wilson     |emma@post.co.uk |London  |2024-03-01 |
|6          |Farid Aliyev    |NULL            |NULL    |2024-03-10 |
|7          |Grace Lee       |grace@web.de    |Berlin  |2024-03-15 |
+-----------+----------------+----------------+--------+-----------+

products
+----------+------------+-----------+------+
|product_id|product_name|category   |price |
+----------+------------+-----------+------+
|101       |Laptop      |Electronics|1200.0|
|102       |Headphones  |Electronics|150.

## Lesson 0 · How a Zillacode problem works

Every PySpark problem on Zillacode opens with this code in the editor:

```python
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = SparkSession.builder.appName('run-pyspark-code').getOrCreate()

def etl(input_df):
	# Write code here
	pass
```

You replace `pass` with code that **returns a DataFrame**. Problems with several inputs pass several DataFrames, like `def etl(customers, orders, products)`. You don't read files and you don't call `show()` for the answer: Zillacode calls your function itself. `F` (the functions) and `W` (the window class) are already imported, and the setup cell of this notebook uses the same names.

**What happens when you press Run.** From Zillacode's source code:
1. Each input is built with `spark.createDataFrame(rows)`, where `rows` is a list of Python dicts such as `{"user_id": 1, "signup": "2023-01-05"}`.
2. Your `etl(...)` is called. The result is turned into a list of rows, and it's compared with the expected rows.
3. Every problem has **2 tests**, and the description shows the data of only the first one. Code that hard-codes values from the example (an id, a date, a count) passes the example and fails the hidden test.

**What step 1 means for the input:**
* Columns arrive in **alphabetical order**, because the dict keys are sorted.
* Whole numbers become `bigint` (long), and decimal numbers become `double`.
* **Dates and timestamps arrive as text**, like `"2023-01-05"`. Convert them with `F.to_date` / `F.to_timestamp` before doing date math (Day 13).

**What step 2 means for your answer:**

| The checker... | so you must... |
|---|---|
| ignores the order of rows | not worry about sorting (sort anyway when the task asks, it never hurts) |
| ignores the order of columns | (nothing) |
| compares column **names** exactly, and every column | return exactly the columns of the output schema, with exactly those names (`alias`), and no leftover helper columns |
| compares **values** exactly: `2` is not `2.0`, `3.14159` is not `3.14` | `cast` to the type of the output schema, and `round` when the task says so |
| treats the texts `None`, `null`, `N/A`, `NaN` as null | (nothing) |

**Spark version.** Zillacode runs Spark 3 with **ANSI mode off**: text that can't become a number or a date turns into null, and dividing by zero gives null. This notebook runs Spark 4.2, where ANSI mode is on by default and the same cases raise errors. The setup cell switches ANSI off, so everything here behaves like Zillacode. (One more small difference: Spark 4 prints a missing value as `NULL`, Spark 3 as `null`.)

In [4]:
# How Zillacode builds an input: a list of dicts
input_df = spark.createDataFrame([
    {"user_id": 1, "name": "Ann", "signup": "2023-01-05", "score": 7.5},
    {"user_id": 2, "name": "Bob", "signup": "2023-02-11", "score": 4.0},
])
input_df.printSchema()      # alphabetical columns; signup is text; user_id is a long
input_df.show()

root
 |-- name: string (nullable = true)
 |-- score: double (nullable = true)
 |-- signup: string (nullable = true)
 |-- user_id: long (nullable = true)

+----+-----+----------+-------+
|name|score|    signup|user_id|
+----+-----+----------+-------+
| Ann|  7.5|2023-01-05|      1|
| Bob|  4.0|2023-02-11|      2|
+----+-----+----------+-------+



In [5]:
# A complete answer: a function that returns a DataFrame
def etl(input_df):
    return input_df.filter(F.col("score") > 5).select("user_id", "name")

result = etl(input_df)
print(type(result).__name__)     # DataFrame: this is what Zillacode needs
result.show()

DataFrame
+-------+----+
|user_id|name|
+-------+----+
|      1| Ann|
+-------+----+



**How to solve a problem, step by step**
1. Read the **output schema** first: these are the columns you must return, with these exact names and types.
2. Compare the example input with the example output and describe in one sentence what happened to the rows: filtered? one row per group? joined with another table? one row per item of a list?
3. Write the steps as separate variables (`joined = ...`, `per_customer = ...`). Here you can `show()` each one; on Zillacode, keep each step small so an error points at one line.
4. Finish with one `select` that names, casts and rounds exactly the output columns.
5. Before you press Run, think about the hidden test: nulls, ties, duplicates, rows without a match, a group with one row.

Documentation for everything in Days 12 to 15: the [functions list](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) and the [DataFrame methods](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html). Keep both open.

## Lesson 1 · Columns: `select`, `withColumn`, expressions and types

A DataFrame never changes in place: every method returns a **new** DataFrame. `df.withColumn(...)` on a line of its own does nothing useful; assign the result (`df = df.withColumn(...)`) or keep chaining.

**Three ways to name a column**

| Form | Use it for |
|---|---|
| `"price"` | a plain name: fine in `select`, `groupBy`, `orderBy` and most `F` functions |
| `F.col("price")` | a Column object, needed for math and comparisons: `F.col("price") * 2`, `F.col("price") > 100` |
| `products["price"]` or `products.price` | the column of one specific DataFrame, needed after a join when both sides have a column with that name (Day 14) |

**The methods you use in almost every answer**

| Method | What it does |
|---|---|
| `df.select(c1, c2, expr.alias("x"))` | returns **only** these columns and expressions: use it to shape the final answer |
| `df.withColumn("x", expr)` | adds column `x`, or **replaces** it if `x` exists; all other columns stay |
| `df.withColumns({"a": e1, "b": e2})` | adds several at once |
| `df.withColumnRenamed("old", "new")` | renames one column |
| `df.drop("a", "b")` | removes columns |
| `expr.alias("x")` | names an expression inside `select` or `agg` |
| `F.lit(5)`, `F.lit("web")` | a constant column |
| `df.selectExpr("price * 2 AS double_price")`, `F.expr("...")` | SQL text instead of Python, when that's shorter |

**Math and types.** `+ - * /` and `%` (remainder) work on columns. `/` **always returns a double**: `7 / 2 = 3.5`. For a whole-number result use `F.floor(a / b)`, or `.cast("int")`, which cuts the decimals off. `F.round(x, 2)` rounds half up; `F.ceil`, `F.floor` and `F.abs` do what their names say. `.cast("int" | "bigint" | "double" | "string" | "date")` changes the type, and text that isn't a number becomes null.

**"The output has the same schema as the input"** (a common Zillacode sentence) means: return the input's columns unchanged. Use `filter` without `select`, or replace a column under the **same name** with `withColumn`.

In [6]:
# select: choose and compute columns; alias names the new ones
products.select(
    "product_name",
    "price",
    (F.col("price") * 1.2).alias("price_with_tax"),
    F.round(F.col("price") * 0.92, 2).alias("price_eur"),
).show()

+------------+------+--------------+---------+
|product_name| price|price_with_tax|price_eur|
+------------+------+--------------+---------+
|      Laptop|1200.0|        1440.0|   1104.0|
|  Headphones| 150.0|         180.0|    138.0|
|Coffee Maker|  85.0|         102.0|     78.2|
|     Blender|  60.0|          72.0|     55.2|
|       Novel|  15.0|          18.0|     13.8|
|    Cookbook|  25.0|          30.0|     23.0|
|   Desk Lamp|  40.0|          48.0|     36.8|
+------------+------+--------------+---------+



In [7]:
# withColumn keeps every column; using an existing name replaces that column
orders.withColumn("quantity", F.col("quantity") * 10).withColumn("source", F.lit("web")).show(4)

+--------+-----------+----------+--------+----------+---------+------+
|order_id|customer_id|product_id|quantity|order_date|   status|source|
+--------+-----------+----------+--------+----------+---------+------+
|    1001|          1|       101|      10|2024-01-10|delivered|   web|
|    1002|          1|       105|      30|2024-01-10|delivered|   web|
|    1003|          2|       102|      20|2024-01-25|delivered|   web|
|    1004|          3|       103|      10|2024-02-03|cancelled|   web|
+--------+-----------+----------+--------+----------+---------+------+
only showing top 4 rows


In [8]:
# Types: / gives a double, cast("int") cuts the decimals, text that is not a number becomes null
spark.range(1).select(
    (F.lit(7) / 2).alias("divide"),
    F.floor(F.lit(7) / 2).alias("floor"),
    (F.lit(7) % 2).alias("remainder"),
    F.lit(3.99).cast("int").alias("cast_int"),
    F.round(F.lit(2.345), 2).alias("round_2"),
    F.lit("12.5").cast("double").alias("text_to_number"),
    F.lit("abc").cast("double").alias("bad_text"),
).show()

+------+-----+---------+--------+-------+--------------+--------+
|divide|floor|remainder|cast_int|round_2|text_to_number|bad_text|
+------+-----+---------+--------+-------+--------------+--------+
|   3.5|    3|        1|       3|   2.35|          12.5|    NULL|
+------+-----+---------+--------+-------+--------------+--------+



In [9]:
# The same kind of thing written as SQL text
products.selectExpr("product_name", "price * 1.2 AS price_with_tax", "CAST(price AS INT) AS whole_price").show(3)

+------------+--------------+-----------+
|product_name|price_with_tax|whole_price|
+------------+--------------+-----------+
|      Laptop|        1440.0|       1200|
|  Headphones|         180.0|        150|
|Coffee Maker|         102.0|         85|
+------------+--------------+-----------+
only showing top 3 rows


### Exercises

**Q1 · `etl(products)`** Show product_name, price and `price_eur` (the price in euros: 1 dollar = 0.92 euro, rounded to 2 decimals), most expensive first.

Expected output:

```
+------------+------+---------+
|product_name| price|price_eur|
+------------+------+---------+
|      Laptop|1200.0|   1104.0|
|  Headphones| 150.0|    138.0|
|Coffee Maker|  85.0|     78.2|
|     Blender|  60.0|     55.2|
|   Desk Lamp|  40.0|     36.8|
|    Cookbook|  25.0|     23.0|
|       Novel|  15.0|     13.8|
+------------+------+---------+
```

In [10]:
products.show()

+----------+------------+-----------+------+
|product_id|product_name|   category| price|
+----------+------------+-----------+------+
|       101|      Laptop|Electronics|1200.0|
|       102|  Headphones|Electronics| 150.0|
|       103|Coffee Maker|    Kitchen|  85.0|
|       104|     Blender|    Kitchen|  60.0|
|       105|       Novel|      Books|  15.0|
|       106|    Cookbook|      Books|  25.0|
|       107|   Desk Lamp|       Home|  40.0|
+----------+------------+-----------+------+



In [11]:
def etl(products):
    products = products.select(
        F.col('product_name'),
        F.col('price'),
        F.round(F.col('price')*0.92,2).alias('price_eur'),
        )
    products = products.orderBy(F.asc('price'))
    return products

etl(products).show()

+------------+------+---------+
|product_name| price|price_eur|
+------------+------+---------+
|       Novel|  15.0|     13.8|
|    Cookbook|  25.0|     23.0|
|   Desk Lamp|  40.0|     36.8|
|     Blender|  60.0|     55.2|
|Coffee Maker|  85.0|     78.2|
|  Headphones| 150.0|    138.0|
|      Laptop|1200.0|   1104.0|
+------------+------+---------+



**Q2 · `etl(products)`** Return the products with the same columns, but with the price as a whole number (int) and the product_name in upper case. Sort by product_id.

Expected output:

```
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      LAPTOP|Electronics| 1200|
|       102|  HEADPHONES|Electronics|  150|
|       103|COFFEE MAKER|    Kitchen|   85|
|       104|     BLENDER|    Kitchen|   60|
|       105|       NOVEL|      Books|   15|
|       106|    COOKBOOK|      Books|   25|
|       107|   DESK LAMP|       Home|   40|
+----------+------------+-----------+-----+
```

In [12]:
def etl(products):
    return (products.withColumn('price',F.col('price').cast('int'))
            .withColumn('product_name',F.upper('product_name'))
            .orderBy('product_id')
     )

# def etl(products):
#     return (products.withColumn("price", F.col("price").cast("int"))
#                     .withColumn("product_name", F.upper("product_name"))
#                     .orderBy("product_id"))



etl(products).show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      LAPTOP|Electronics| 1200|
|       102|  HEADPHONES|Electronics|  150|
|       103|COFFEE MAKER|    Kitchen|   85|
|       104|     BLENDER|    Kitchen|   60|
|       105|       NOVEL|      Books|   15|
|       106|    COOKBOOK|      Books|   25|
|       107|   DESK LAMP|       Home|   40|
+----------+------------+-----------+-----+



## Lesson 2 · Filtering rows

`filter(condition)` and `where(condition)` are the same method. A condition is a column of true/false values:

| Condition | Meaning |
|---|---|
| `F.col("qty") >= 2`, `==`, `!=`, `<`, `>` | comparisons (`==`, not `=`) |
| `(c1) & (c2)`, `(c1) \| (c2)`, `~(c1)` | and, or, not: **put each part in parentheses** |
| `F.col("status").isin("shipped", "cancelled")` | one of several values |
| `F.col("price").between(50, 200)` | a range, **both ends included** |
| `F.col("email").isNull()`, `.isNotNull()` | missing / present |
| `.startswith("A")`, `.endswith(".uz")`, `.contains("@")` | text tests |
| `.like("A%")` | SQL pattern: `%` is any text, `_` is one character |
| `.rlike("^[0-9]+$")` | a regular expression (Lesson 6) |
| `"price > 100 AND category = 'Books'"` | a SQL string works too |

**Why the parentheses.** In Python, `&` binds tighter than `>`, so `F.col("a") > 1 & F.col("b") < 5` is read as `F.col("a") > (1 & F.col("b")) < 5` and fails. Also, never use Python's `and` / `or` / `not` on columns: Spark can't turn a column into one True/False and stops with *Cannot convert column into bool*.

**Nulls in filters.** A comparison with null gives null ("unknown"), and `filter` keeps only the rows where the condition is **true**. So `F.col("city") != "Berlin"` also drops every customer whose city is null. If they belong in the answer, say so: `(F.col("city") != "Berlin") | F.col("city").isNull()`. This is one of the most common reasons a Zillacode answer passes the example and fails the hidden test.

In [13]:
orders.filter((F.col("status") == "delivered") & (F.col("quantity") >= 3)).show()

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1002|          1|       105|       3|2024-01-10|delivered|
|    1009|          3|       105|       4|2024-03-05|delivered|
|    1014|          5|       105|       5|2024-03-20|delivered|
+--------+-----------+----------+--------+----------+---------+



In [14]:
products.where(F.col("category").isin("Books", "Home") | (F.col("price") > 1000)).show()

+----------+------------+-----------+------+
|product_id|product_name|   category| price|
+----------+------------+-----------+------+
|       101|      Laptop|Electronics|1200.0|
|       105|       Novel|      Books|  15.0|
|       106|    Cookbook|      Books|  25.0|
|       107|   Desk Lamp|       Home|  40.0|
+----------+------------+-----------+------+



In [15]:
# != drops the null city: Farid (customer 6) is missing here...
customers.filter(F.col("city") != "Berlin").select("customer_id", "name", "city").show()
# ...and he comes back when you ask for the nulls too
customers.filter((F.col("city") != "Berlin") | F.col("city").isNull()).select("customer_id", "name", "city").show()

+-----------+----------------+--------+
|customer_id|            name|    city|
+-----------+----------------+--------+
|          1|    Aziz Karimov|Tashkent|
|          3|     Carlos Ruiz|  Madrid|
|          4|Dilnoza Rahimova|Tashkent|
|          5|     Emma Wilson|  London|
+-----------+----------------+--------+

+-----------+----------------+--------+
|customer_id|            name|    city|
+-----------+----------------+--------+
|          1|    Aziz Karimov|Tashkent|
|          3|     Carlos Ruiz|  Madrid|
|          4|Dilnoza Rahimova|Tashkent|
|          5|     Emma Wilson|  London|
|          6|    Farid Aliyev|    NULL|
+-----------+----------------+--------+



In [16]:
# Python's `and` doesn't work on columns
try:
    orders.filter(F.col("quantity") > 1 and F.col("status") == "delivered")
except Exception as e:
    print(type(e).__name__, "->", str(e).split(".")[0])

PySparkValueError -> [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions


### Exercises

**Q3 · `etl(orders)`** Show the orders that are shipped or cancelled and have a quantity of 2 or more: order_id, status and quantity, sorted by order_id.

Expected output:

```
+--------+---------+--------+
|order_id|   status|quantity|
+--------+---------+--------+
|    1006|  shipped|       2|
|    1010|cancelled|       2|
|    1012|  shipped|       2|
+--------+---------+--------+
```

In [24]:
def etl(orders):
    orders=orders.filter(((F.col('status')=='shipped') | (F.col('status')=='cancelled')) & (F.col('quantity')>=2))
    orders=orders.orderBy(F.asc('order_id'))
    return orders

etl(orders).show()

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1006|          4|       104|       2|2024-02-15|  shipped|
|    1010|          4|       106|       2|2024-03-08|cancelled|
|    1012|          6|       103|       2|2024-03-12|  shipped|
+--------+-----------+----------+--------+----------+---------+



**Q4 · `etl(customers)`** Show the customers whose email is from Uzbekistan or Germany (it ends with `.uz` or `.de`): name and email, sorted by name.

Expected output:

```
+----------------+---------------+
|            name|          email|
+----------------+---------------+
|    Aziz Karimov|   aziz@mail.uz|
|   Bella Schmidt|   bella@web.de|
|Dilnoza Rahimova|dilnoza@mail.uz|
|       Grace Lee|   grace@web.de|
+----------------+---------------+
```

In [25]:
def etl(customers):
    customers=customers.filter(F.col('email').endswith('uz') | F.col('email').endswith('de'))

    return customers

etl(customers).show()

+-----------+----------------+---------------+--------+-----------+
|customer_id|            name|          email|    city|signup_date|
+-----------+----------------+---------------+--------+-----------+
|          1|    Aziz Karimov|   aziz@mail.uz|Tashkent| 2024-01-05|
|          2|   Bella Schmidt|   bella@web.de|  Berlin| 2024-01-20|
|          4|Dilnoza Rahimova|dilnoza@mail.uz|Tashkent| 2024-02-14|
|          7|       Grace Lee|   grace@web.de|  Berlin| 2024-03-15|
+-----------+----------------+---------------+--------+-----------+



**Q5 · `etl(customers)`** Show every customer who doesn't live in Berlin, customers without a city included: customer_id, name and city, sorted by customer_id.

Expected output:

```
+-----------+----------------+--------+
|customer_id|            name|    city|
+-----------+----------------+--------+
|          1|    Aziz Karimov|Tashkent|
|          3|     Carlos Ruiz|  Madrid|
|          4|Dilnoza Rahimova|Tashkent|
|          5|     Emma Wilson|  London|
|          6|    Farid Aliyev|    NULL|
+-----------+----------------+--------+
```

In [29]:
def etl(customers):
    customers=customers.filter((customers.city!="Berlin") | F.col('city').isNull())
    return customers.orderBy(F.desc('customer_id'))

etl(customers).show()

+-----------+----------------+----------------+--------+-----------+
|customer_id|            name|           email|    city|signup_date|
+-----------+----------------+----------------+--------+-----------+
|          6|    Farid Aliyev|            NULL|    NULL| 2024-03-10|
|          5|     Emma Wilson| emma@post.co.uk|  London| 2024-03-01|
|          4|Dilnoza Rahimova| dilnoza@mail.uz|Tashkent| 2024-02-14|
|          3|     Carlos Ruiz|carlos@correo.es|  Madrid| 2024-02-02|
|          1|    Aziz Karimov|    aziz@mail.uz|Tashkent| 2024-01-05|
+-----------+----------------+----------------+--------+-----------+



## Lesson 3 · Conditional logic: `F.when(...).otherwise(...)`

This is Spark's `if / elif / else` for a column (SQL calls it `CASE WHEN`):

```python
F.when(condition1, value1).when(condition2, value2).otherwise(default)
```

* The conditions are checked **top to bottom for each row**, and the **first true one wins**. So go from the most specific to the most general: `when(price > 1000, "luxury").when(price > 100, "premium")`. In the other order, nothing would ever be "luxury".
* Without `.otherwise(...)`, a row that matches nothing gets **null**.
* A value can be a constant (`"cheap"`, `0`) or a column expression (`F.col("price") * 0.9`). Keep all branches of the same type.
* A condition can combine several columns: `(F.col("status") == "active") & (F.col("country") == "USA")`.
* For a plain true/false column you don't need `when` at all: `F.col("quantity") >= 3` already **is** a boolean column.

On Zillacode, problem #16 pays overtime (a formula inside `otherwise`), #33 builds a label from two columns, and #41 avoids dividing by zero.

In [30]:
# Order the conditions from the most specific
orders.select(
    "order_id", "status", "quantity",
    F.when(F.col("status") == "cancelled", "none")
     .when(F.col("quantity") >= 3, "large")
     .when(F.col("quantity") == 2, "medium")
     .otherwise("small").alias("size"),
).show(6)

+--------+---------+--------+------+
|order_id|   status|quantity|  size|
+--------+---------+--------+------+
|    1001|delivered|       1| small|
|    1002|delivered|       3| large|
|    1003|delivered|       2|medium|
|    1004|cancelled|       1|  none|
|    1005|delivered|       1| small|
|    1006|  shipped|       2|medium|
+--------+---------+--------+------+
only showing top 6 rows


In [31]:
# A branch can be a formula; without otherwise, unmatched rows get null
products.select(
    "product_name", "category", "price",
    F.when(F.col("category") == "Books", F.round(F.col("price") * 0.8, 2)).otherwise(F.col("price")).alias("sale_price"),
    F.when(F.col("price") > 1000, "check stock").alias("note"),
).show()

+------------+-----------+------+----------+-----------+
|product_name|   category| price|sale_price|       note|
+------------+-----------+------+----------+-----------+
|      Laptop|Electronics|1200.0|    1200.0|check stock|
|  Headphones|Electronics| 150.0|     150.0|       NULL|
|Coffee Maker|    Kitchen|  85.0|      85.0|       NULL|
|     Blender|    Kitchen|  60.0|      60.0|       NULL|
|       Novel|      Books|  15.0|      12.0|       NULL|
|    Cookbook|      Books|  25.0|      20.0|       NULL|
|   Desk Lamp|       Home|  40.0|      40.0|       NULL|
+------------+-----------+------+----------+-----------+



In [32]:
# Safe division: when the divisor is 0, return 0 instead of null
spark.createDataFrame([(10.0, 4), (5.0, 0)], "total double, n int").select(
    "total", "n",
    (F.col("total") / F.col("n")).alias("plain_divide"),
    F.when(F.col("n") != 0, F.round(F.col("total") / F.col("n"), 2)).otherwise(0).alias("safe_divide"),
).show()

+-----+---+------------+-----------+
|total|  n|plain_divide|safe_divide|
+-----+---+------------+-----------+
| 10.0|  4|         2.5|        2.5|
|  5.0|  0|        NULL|        0.0|
+-----+---+------------+-----------+



### Exercises

**Q6 · `etl(products)`** Give every product a `price_level`: `cheap` if the price is under 50, `mid` from 50 to 200, `premium` above 200. Show product_name, price and price_level, sorted by price.

Expected output:

```
+------------+------+-----------+
|product_name| price|price_level|
+------------+------+-----------+
|       Novel|  15.0|      cheap|
|    Cookbook|  25.0|      cheap|
|   Desk Lamp|  40.0|      cheap|
|     Blender|  60.0|        mid|
|Coffee Maker|  85.0|        mid|
|  Headphones| 150.0|        mid|
|      Laptop|1200.0|    premium|
+------------+------+-----------+
```

In [34]:
def etl(products):
    products=products.select(
        'product_name','price',
        F.when(F.col('price')<50,'cheap')
        .when(((F.col('price')>50) & (F.col('price')<200)),'mid')
        .otherwise('premium').alias('price_leve')
    )

    return products

etl(products).show()

+------------+------+----------+
|product_name| price|price_leve|
+------------+------+----------+
|      Laptop|1200.0|   premium|
|  Headphones| 150.0|       mid|
|Coffee Maker|  85.0|       mid|
|     Blender|  60.0|       mid|
|       Novel|  15.0|     cheap|
|    Cookbook|  25.0|     cheap|
|   Desk Lamp|  40.0|     cheap|
+------------+------+----------+



**Q7 · `etl(orders)`** Label every order: `problem` if it was cancelled, otherwise `guest` if it has no customer_id, otherwise `big` if the quantity is 3 or more, otherwise `normal`. Show order_id and label, sorted by order_id.

Expected output:

```
+--------+-------+
|order_id|  label|
+--------+-------+
|    1001| normal|
|    1002|    big|
|    1003| normal|
|    1004|problem|
|    1005| normal|
|    1006| normal|
|    1007| normal|
|    1008| normal|
|    1009|    big|
|    1010|problem|
|    1011|  guest|
|    1012| normal|
|    1013| normal|
|    1014|    big|
|    1015| normal|
|    1016| normal|
+--------+-------+
```

In [36]:
def etl(orders):
    orders=orders.select(
        'order_id',
        F.when(F.col('status')=='cancelled','problem')
        .when(F.col('customer_id').isNull(),'geust')
        .when(F.col('quantity')>=3,'big')
        .otherwise('normal').alias('label')
    )

    return orders


etl(orders).show()

+--------+-------+
|order_id|  label|
+--------+-------+
|    1001| normal|
|    1002|    big|
|    1003| normal|
|    1004|problem|
|    1005| normal|
|    1006| normal|
|    1007| normal|
|    1008| normal|
|    1009|    big|
|    1010|problem|
|    1011|  geust|
|    1012| normal|
|    1013| normal|
|    1014|    big|
|    1015| normal|
|    1016| normal|
+--------+-------+



## Lesson 4 · Missing values (null)

Null means "unknown". Three rules cause most of the wrong answers:
1. **Comparisons with null give null**, and `filter` drops those rows (Lesson 2). Even `null == null` is null, not true. To compare two columns that may both be null, use `a.eqNullSafe(b)` (SQL `<=>`): it's true when both are null.
2. **Math with null gives null**: `5 + null` is null. `F.concat("a", null)` is null too, but `F.concat_ws(" ", "a", null)` skips the null and gives `"a"`.
3. **Aggregates skip nulls**: `sum`, `avg` and `count("col")` ignore them, but `count("*")` counts every row (Day 13).

| Tool | What it does |
|---|---|
| `F.col("x").isNull()`, `.isNotNull()`, `F.isnull("x")` | test for null |
| `F.coalesce(c1, c2, F.lit(default))` | the first non-null value, left to right |
| `df.fillna(0)`, `df.na.fill({"city": "Unknown", "qty": 0})` | replace nulls. **A single value only fills columns of its own type**: `fillna(0)` fills number columns, `fillna("x")` fills text columns. A dict sets a value per column. |
| `df.dropna()`, `df.dropna(how="all")`, `df.dropna(subset=["email"])` | drop rows with a null in any column / in every column / in the chosen columns |
| `F.when(F.col("x") != "", F.col("x"))` | turn an empty text into null (no `otherwise`, so the rest becomes null) |
| `a.eqNullSafe(b)` | equality where null equals null |

On Zillacode: #11 keeps the rows with nulls, #10 and #3 turn nulls into 0 after joins and pivots, #14 drops rows where every column is null, and #26 fills a missing value.

In [37]:
customers.select(
    "customer_id", "email", "city",
    F.col("email").isNull().alias("no_email"),
    F.coalesce("city", F.lit("Unknown")).alias("city_or_unknown"),
    F.concat("name", F.lit(" from "), "city").alias("with_concat"),
    F.concat_ws(" from ", "name", "city").alias("with_concat_ws"),
).show(truncate=False)

+-----------+----------------+--------+--------+---------------+------------------------------+------------------------------+
|customer_id|email           |city    |no_email|city_or_unknown|with_concat                   |with_concat_ws                |
+-----------+----------------+--------+--------+---------------+------------------------------+------------------------------+
|1          |aziz@mail.uz    |Tashkent|false   |Tashkent       |Aziz Karimov from Tashkent    |Aziz Karimov from Tashkent    |
|2          |bella@web.de    |Berlin  |false   |Berlin         |Bella Schmidt from Berlin     |Bella Schmidt from Berlin     |
|3          |carlos@correo.es|Madrid  |false   |Madrid         |Carlos Ruiz from Madrid       |Carlos Ruiz from Madrid       |
|4          |dilnoza@mail.uz |Tashkent|false   |Tashkent       |Dilnoza Rahimova from Tashkent|Dilnoza Rahimova from Tashkent|
|5          |emma@post.co.uk |London  |false   |London         |Emma Wilson from London       |Emma Wilson from

In [38]:
# fillna with a dict: one value per column
customers.fillna({"email": "no email", "city": "Unknown"}).filter(F.col("customer_id") >= 5).show()

+-----------+------------+---------------+-------+-----------+
|customer_id|        name|          email|   city|signup_date|
+-----------+------------+---------------+-------+-----------+
|          5| Emma Wilson|emma@post.co.uk| London| 2024-03-01|
|          6|Farid Aliyev|       no email|Unknown| 2024-03-10|
|          7|   Grace Lee|   grace@web.de| Berlin| 2024-03-15|
+-----------+------------+---------------+-------+-----------+



In [39]:
# null == null is not true; eqNullSafe is
spark.createDataFrame([("a", "a"), ("a", None), (None, None)], "x string, y string").select(
    "x", "y",
    (F.col("x") == F.col("y")).alias("x == y"),
    F.col("x").eqNullSafe(F.col("y")).alias("x <=> y"),
).show()

+----+----+------+-------+
|   x|   y|x == y|x <=> y|
+----+----+------+-------+
|   a|   a|  true|   true|
|   a|NULL|  NULL|  false|
|NULL|NULL|  NULL|   true|
+----+----+------+-------+



In [40]:
# dropna: any null (default), every column null, or only some columns
t = spark.createDataFrame([(1, "a"), (2, None), (None, None)], "id int, v string")
t.dropna().show()
t.dropna(how="all").show()
t.dropna(subset=["id"]).show()

+---+---+
| id|  v|
+---+---+
|  1|  a|
+---+---+

+---+----+
| id|   v|
+---+----+
|  1|   a|
|  2|NULL|
+---+----+

+---+----+
| id|   v|
+---+----+
|  1|   a|
|  2|NULL|
+---+----+



### Exercises

**Q8 · `etl(customers)`** Show customer_id, name and city of every customer, with `Unknown` where the city is missing. Call the name column `customer_name`. Sort by customer_id.

Expected output:

```
+-----------+----------------+--------+
|customer_id|   customer_name|    city|
+-----------+----------------+--------+
|          1|    Aziz Karimov|Tashkent|
|          2|   Bella Schmidt|  Berlin|
|          3|     Carlos Ruiz|  Madrid|
|          4|Dilnoza Rahimova|Tashkent|
|          5|     Emma Wilson|  London|
|          6|    Farid Aliyev| Unknown|
|          7|       Grace Lee|  Berlin|
+-----------+----------------+--------+
```

In [47]:
def etl(customers):
    customers=customers.fillna({'city':'UNKNOWN'})
    return customers

etl(customers).show()

+-----------+----------------+----------------+--------+-----------+
|customer_id|            name|           email|    city|signup_date|
+-----------+----------------+----------------+--------+-----------+
|          1|    Aziz Karimov|    aziz@mail.uz|Tashkent| 2024-01-05|
|          2|   Bella Schmidt|    bella@web.de|  Berlin| 2024-01-20|
|          3|     Carlos Ruiz|carlos@correo.es|  Madrid| 2024-02-02|
|          4|Dilnoza Rahimova| dilnoza@mail.uz|Tashkent| 2024-02-14|
|          5|     Emma Wilson| emma@post.co.uk|  London| 2024-03-01|
|          6|    Farid Aliyev|            NULL| UNKNOWN| 2024-03-10|
|          7|       Grace Lee|    grace@web.de|  Berlin| 2024-03-15|
+-----------+----------------+----------------+--------+-----------+



**Q9 · `etl(signups_raw)`** An empty email (`""`) means "no email". Show signup_id, `email` (null instead of an empty text) and `has_email` (true or false), sorted by signup_id.

Expected output:

```
+---------+------------------+---------+
|signup_id|             email|has_email|
+---------+------------------+---------+
|        1|     Aziz@Mail.UZ |     true|
|        2|      bella@web.de|     true|
|        3|              NULL|    false|
|        4|   dilnoza@mail.uz|     true|
|        5|emma[at]post.co.uk|     true|
|        6|      aziz@mail.uz|     true|
+---------+------------------+---------+
```

In [54]:
def etl(signups_raw):

    signups_raw=signups_raw.select(
        'signup_id','email',
        F.when(
            (F.col('email').isNull()) | (F.col('email')==""), False)
            .otherwise(True)
            .alias('has_email')
    )

    return signups_raw

etl(signups_raw).show()

+---------+------------------+---------+
|signup_id|             email|has_email|
+---------+------------------+---------+
|        1|     Aziz@Mail.UZ |     true|
|        2|      bella@web.de|     true|
|        3|                  |    false|
|        4|   dilnoza@mail.uz|     true|
|        5|emma[at]post.co.uk|     true|
|        6|      aziz@mail.uz|     true|
+---------+------------------+---------+



## Lesson 5 · Text functions

| Function | Example and result |
|---|---|
| `F.upper`, `F.lower`, `F.initcap` | `initcap("aziz KARIMOV")` gives `Aziz Karimov` |
| `F.trim`, `F.ltrim`, `F.rtrim` | `trim("  hi ")` gives `hi` (spaces at the ends only) |
| `F.length` | `length("Spark")` gives 5 |
| `F.concat(a, b, ...)` | glues texts; null if any part is null |
| `F.concat_ws(sep, a, b, ...)` | glues with a separator and skips nulls |
| `F.substring(col, pos, len)` | **counts from 1**: `substring("Spark", 1, 3)` gives `Spa`; a negative pos counts from the end: `substring("Spark", -2, 2)` gives `rk` |
| `F.split(col, pattern)` | text to an **array**; the pattern is a regular expression, so escape `.` and `\|` as `"\\."` and `"\\\|"` |
| `arr[0]`, `arr.getItem(0)`, `F.element_at(arr, -1)` | one item: `[ ]` counts from **0**; `element_at` counts from **1** and takes -1 for the last item |
| `F.size(arr)` | the number of items |
| `F.substring_index(col, "@", -1)` | the text after the last `@`; with `1`, the text before the first `@` |
| `F.instr(col, "x")` | the position of a substring, counted from 1; 0 if it's not there |
| `F.lpad(col, 5, "0")`, `F.rpad` | pad to a length: `lpad("42", 5, "0")` gives `00042` |
| `F.translate(col, "abc", "xyz")` | replaces characters one by one |
| `F.regexp_replace`, `F.regexp_extract` | patterns, Lesson 6 |

**Two traps:**
* Counting the items of a list-like text with `F.size(F.split(col, ","))`: a **null** text gives **-1**, and an **empty** text gives **1** (one empty item). Guard both: `F.when(F.col("x").isNull() | (F.col("x") == ""), 0).otherwise(F.size(F.split("x", ",")))`.
* Integer columns used in text functions: Spark converts them for you, but cast explicitly (`F.col("id").cast("string")`) when you glue numbers into a code, to keep the intent clear.

On Zillacode: #2 builds a full name with `concat_ws`, #48 splits names and "type, color" texts, #49 measures lengths, #50 upper-cases, and #40 counts the items of comma-separated lists.

In [55]:
customers.select(
    "name",
    F.upper("name").alias("upper"),
    F.length("name").alias("len"),
    F.split("name", " ").alias("parts"),
    F.split("name", " ")[0].alias("first"),
    F.element_at(F.split("name", " "), -1).alias("last"),
    F.substring("name", 1, 3).alias("first_3"),
).show(truncate=False)

+----------------+----------------+---+-------------------+-------+--------+-------+
|name            |upper           |len|parts              |first  |last    |first_3|
+----------------+----------------+---+-------------------+-------+--------+-------+
|Aziz Karimov    |AZIZ KARIMOV    |12 |[Aziz, Karimov]    |Aziz   |Karimov |Azi    |
|Bella Schmidt   |BELLA SCHMIDT   |13 |[Bella, Schmidt]   |Bella  |Schmidt |Bel    |
|Carlos Ruiz     |CARLOS RUIZ     |11 |[Carlos, Ruiz]     |Carlos |Ruiz    |Car    |
|Dilnoza Rahimova|DILNOZA RAHIMOVA|16 |[Dilnoza, Rahimova]|Dilnoza|Rahimova|Dil    |
|Emma Wilson     |EMMA WILSON     |11 |[Emma, Wilson]     |Emma   |Wilson  |Emm    |
|Farid Aliyev    |FARID ALIYEV    |12 |[Farid, Aliyev]    |Farid  |Aliyev  |Far    |
|Grace Lee       |GRACE LEE       |9  |[Grace, Lee]       |Grace  |Lee     |Gra    |
+----------------+----------------+---+-------------------+-------+--------+-------+



In [56]:
customers.select(
    "email",
    F.substring_index("email", "@", 1).alias("user"),
    F.substring_index("email", "@", -1).alias("domain"),
    F.instr("email", "@").alias("at_position"),
    F.lpad(F.col("customer_id").cast("string"), 4, "0").alias("padded_id"),
).show(truncate=False)

+----------------+-------+----------+-----------+---------+
|email           |user   |domain    |at_position|padded_id|
+----------------+-------+----------+-----------+---------+
|aziz@mail.uz    |aziz   |mail.uz   |5          |0001     |
|bella@web.de    |bella  |web.de    |6          |0002     |
|carlos@correo.es|carlos |correo.es |7          |0003     |
|dilnoza@mail.uz |dilnoza|mail.uz   |8          |0004     |
|emma@post.co.uk |emma   |post.co.uk|5          |0005     |
|NULL            |NULL   |NULL      |NULL       |0006     |
|grace@web.de    |grace  |web.de    |6          |0007     |
+----------------+-------+----------+-----------+---------+



In [ ]:
# Counting the items of a list-like text: the null row gives -1 until you guard it
reviews.select(
    "review_id", "tags",
    F.size(F.split("tags", ";")).alias("naive_count"),
    F.when(F.col("tags").isNull() | (F.col("tags") == ""), 0).otherwise(F.size(F.split("tags", ";"))).alias("tag_count"),
).show(truncate=False)

+---------+------------------------+-----------+---------+
|review_id|tags                    |naive_count|tag_count|
+---------+------------------------+-----------+---------+
|1        |fast delivery;good price|2          |2        |
|2        |good price              |1          |1        |
|3        |slow delivery           |1          |1        |
|4        |broken;slow delivery    |2          |2        |
|5        |fast delivery           |1          |1        |
|6        |NULL                    |-1         |0        |
|7        |good price;gift         |2          |2        |
|8        |gift                    |1          |1        |
+---------+------------------------+-----------+---------+


In [ ]:
# Messy names: trim the ends, collapse inner spaces, fix the case
signups_raw.select(
    "full_name",
    F.initcap(F.trim("full_name")).alias("trim_initcap"),
    F.initcap(F.trim(F.regexp_replace("full_name", r"\s+", " "))).alias("clean"),
).show(truncate=False)

+-----------------+----------------+----------------+
|full_name        |trim_initcap    |clean           |
+-----------------+----------------+----------------+
|  aziz   KARIMOV |Aziz   Karimov  |Aziz Karimov    |
|bella schmidt    |Bella Schmidt   |Bella Schmidt   |
|Carlos  Ruiz     |Carlos  Ruiz    |Carlos Ruiz     |
|DILNOZA rahimova |Dilnoza Rahimova|Dilnoza Rahimova|
|Emma Wilson      |Emma Wilson     |Emma Wilson     |
|aziz karimov     |Aziz Karimov    |Aziz Karimov    |
+-----------------+----------------+----------------+


### Exercises

**Q10 · `etl(customers)`** Split `name` into `first_name` and `last_name`. Show customer_id, first_name and last_name, sorted by customer_id.

Expected output:

```
+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
|          1|      Aziz|  Karimov|
|          2|     Bella|  Schmidt|
|          3|    Carlos|     Ruiz|
|          4|   Dilnoza| Rahimova|
|          5|      Emma|   Wilson|
|          6|     Farid|   Aliyev|
|          7|     Grace|      Lee|
+-----------+----------+---------+
```

In [57]:
customers.show()

+-----------+----------------+----------------+--------+-----------+
|customer_id|            name|           email|    city|signup_date|
+-----------+----------------+----------------+--------+-----------+
|          1|    Aziz Karimov|    aziz@mail.uz|Tashkent| 2024-01-05|
|          2|   Bella Schmidt|    bella@web.de|  Berlin| 2024-01-20|
|          3|     Carlos Ruiz|carlos@correo.es|  Madrid| 2024-02-02|
|          4|Dilnoza Rahimova| dilnoza@mail.uz|Tashkent| 2024-02-14|
|          5|     Emma Wilson| emma@post.co.uk|  London| 2024-03-01|
|          6|    Farid Aliyev|            NULL|    NULL| 2024-03-10|
|          7|       Grace Lee|    grace@web.de|  Berlin| 2024-03-15|
+-----------+----------------+----------------+--------+-----------+



In [60]:
customers.select(F.split(customers.name,' ')[0]).show()

+---------------------+
|split(name,  , -1)[0]|
+---------------------+
|                 Aziz|
|                Bella|
|               Carlos|
|              Dilnoza|
|                 Emma|
|                Farid|
|                Grace|
+---------------------+



In [62]:
def etl(customers):
    customers=customers.select(
        'customer_id',
        F.split('name',' ')[0].alias('first_name'),
        F.split('name',' ')[1].alias('last_name')
                     )
    return customers

etl(customers).show()

+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
|          1|      Aziz|  Karimov|
|          2|     Bella|  Schmidt|
|          3|    Carlos|     Ruiz|
|          4|   Dilnoza| Rahimova|
|          5|      Emma|   Wilson|
|          6|     Farid|   Aliyev|
|          7|     Grace|      Lee|
+-----------+----------+---------+



**Q11 · `etl(customers)`** Make a short name for every customer: the first name, a space, the first letter of the last name and a dot, like `Aziz K.` Show customer_id and short_name, sorted by customer_id.

Expected output:

```
+-----------+----------+
|customer_id|short_name|
+-----------+----------+
|          1|   Aziz K.|
|          2|  Bella S.|
|          3| Carlos R.|
|          4|Dilnoza R.|
|          5|   Emma W.|
|          6|  Farid A.|
|          7|  Grace L.|
+-----------+----------+
```

In [72]:
def etl(customers):
    customers=customers.select(
        "customer_id",
        F.concat(F.split('name'," ")[0],
                 F.lit(' '),
                 (F.substring(F.split('name',' ')[1],1,1)),
                 F.lit('.')).alias('short_name')
    )

    return customers

etl(customers).show()

+-----------+----------+
|customer_id|short_name|
+-----------+----------+
|          1|   Aziz K.|
|          2|  Bella S.|
|          3| Carlos R.|
|          4|Dilnoza R.|
|          5|   Emma W.|
|          6|  Farid A.|
|          7|  Grace L.|
+-----------+----------+



**Q12 · `etl(customers)`** Mask the emails: keep the first letter, then `***`, then everything from the `@` on (like `a***@mail.uz`). Customers without an email drop out. Show name and masked_email, sorted by name.

Expected output:

```
+----------------+---------------+
|            name|   masked_email|
+----------------+---------------+
|    Aziz Karimov|   a***@mail.uz|
|   Bella Schmidt|    b***@web.de|
|     Carlos Ruiz| c***@correo.es|
|Dilnoza Rahimova|   d***@mail.uz|
|     Emma Wilson|e***@post.co.uk|
|       Grace Lee|    g***@web.de|
+----------------+---------------+
```

In [79]:
def etl(customers):
    customers=customers.select(
        'name',
        F.concat(
            F.substring(F.split('email','@')[0],1,1),
            F.lit('****@'),
            F.split('email','@')[1]
        ).alias('masked_email')
    ).filter('masked_email is not null')

    return customers

etl(customers).show()

+----------------+----------------+
|            name|    masked_email|
+----------------+----------------+
|    Aziz Karimov|   a****@mail.uz|
|   Bella Schmidt|    b****@web.de|
|     Carlos Ruiz| c****@correo.es|
|Dilnoza Rahimova|   d****@mail.uz|
|     Emma Wilson|e****@post.co.uk|
|       Grace Lee|    g****@web.de|
+----------------+----------------+



**Q13 · `etl(products)`** Give every product a `code`: the first 3 letters of its category in upper case, a dash, and the product_id, like `ELE-101`. Show product_name and code, sorted by code.

Expected output:

```
+------------+-------+
|product_name|   code|
+------------+-------+
|       Novel|BOO-105|
|    Cookbook|BOO-106|
|      Laptop|ELE-101|
|  Headphones|ELE-102|
|   Desk Lamp|HOM-107|
|Coffee Maker|KIT-103|
|     Blender|KIT-104|
+------------+-------+
```

In [84]:
def etl(products):
    products=products.select(
        'product_name',
        F.concat(
            F.upper(F.substring('category',1,3)),
            F.lit('-'),
            F.col('product_id')
        ).alias('code')
    )

    return products.orderBy('code')

etl(products).show()

+------------+-------+
|product_name|   code|
+------------+-------+
|       Novel|BOO-105|
|    Cookbook|BOO-106|
|      Laptop|ELE-101|
|  Headphones|ELE-102|
|   Desk Lamp|HOM-107|
|Coffee Maker|KIT-103|
|     Blender|KIT-104|
+------------+-------+



## Lesson 6 · Regular expressions

Three functions do almost everything:

| Function | What it does |
|---|---|
| `col.rlike(pattern)` | true if the pattern matches **anywhere** in the text. Wrap it in `^...$` to make the **whole** text match. |
| `F.regexp_extract(col, pattern, group)` | the matched part: group `0` is the whole match, `1` is the first `( )` group. **No match gives an empty text `""`, not null.** |
| `F.regexp_replace(col, pattern, replacement)` | replaces **every** match |

**Pattern cheat sheet** (Spark uses [Java's regex syntax](https://docs.oracle.com/javase/8/docs/api/java/util/regex/Pattern.html)):

| Piece | Matches |
|---|---|
| `\d`, `\D` | a digit, anything but a digit |
| `\s`, `\w` | a space or tab; a letter, digit or `_` |
| `.` and `\.` | any character; a real dot |
| `[abc]`, `[^abc]`, `[0-9a-z]` | one of these; anything but these; a range |
| `+`, `*`, `?` | 1 or more, 0 or more, optional |
| `{3}`, `{2,4}` | exactly 3, 2 to 4 |
| `^`, `$` | the start, the end of the text |
| `( )`, `a\|b` | a group to extract; a or b |

Write patterns as **raw strings**, `r"\d+"`, so Python leaves the backslashes alone.

On Zillacode: #4 extracts the email domain and masks the first 6 digits of a phone, #6 replaces a word, #17 extracts the number from a description, #26 extracts a `[10% off]` discount, #30 checks that a date text looks like `yyyy-MM-dd`, and #51 keeps valid arithmetic expressions.

In [85]:
# looks_valid needs the WHOLE text to match (^...$): row 1 fails only because of the space at its end
signups_raw.select(
    "email",
    F.col("email").rlike("@").alias("has_at"),
    F.col("email").rlike(r"^[\w.]+@[\w.]+\.[A-Za-z]{2,}$").alias("looks_valid"),
    F.regexp_extract("email", r"@(.+)$", 1).alias("after_at"),
).show(truncate=False)

+------------------+------+-----------+--------+
|email             |has_at|looks_valid|after_at|
+------------------+------+-----------+--------+
|Aziz@Mail.UZ      |true  |false      |Mail.UZ |
|bella@web.de      |true  |true       |web.de  |
|                  |false |false      |        |
|dilnoza@mail.uz   |true  |true       |mail.uz |
|emma[at]post.co.uk|false |false      |        |
|aziz@mail.uz      |true  |true       |mail.uz |
+------------------+------+-----------+--------+



In [ ]:
# Group 0 vs group 1, and "" when nothing matches
texts = spark.createDataFrame([("Order #123 shipped",), ("no number here",), ("[15% off] summer sale",)], "text string")
first_number = F.regexp_extract("text", r"\d+", 0)
texts.select(
    "text",
    first_number.alias("first_number"),
    F.regexp_extract("text", r"\[(\d+)% off\]", 1).alias("discount"),
    F.when(first_number != "", first_number.cast("int")).alias("number_or_null"),
).show(truncate=False)

+---------------------+------------+--------+--------------+
|text                 |first_number|discount|number_or_null|
+---------------------+------------+--------+--------------+
|Order #123 shipped   |123         |        |123           |
|no number here       |            |        |NULL          |
|[15% off] summer sale|15          |15      |15            |
+---------------------+------------+--------+--------------+


In [86]:
# regexp_replace: clean or mask
signups_raw.select(
    "phone",
    F.regexp_replace("phone", r"[^0-9]", "").alias("digits_only"),
    F.regexp_replace(F.regexp_replace("phone", r"[^0-9]", ""), r"^\d{5}", "*****").alias("masked"),
).show(truncate=False)

+-----------------+--------------+--------------+
|phone            |digits_only   |masked        |
+-----------------+--------------+--------------+
|+998 90 123-45-67|998901234567  |*****1234567  |
|0049 151 2345678 |00491512345678|*****512345678|
|(34) 612 345 678 |34612345678   |*****345678   |
|90-765-43-21     |907654321     |*****4321     |
|07700 900123     |07700900123   |*****900123   |
|+998901234567    |998901234567  |*****1234567  |
+-----------------+--------------+--------------+



### Exercises

**Q14 · `etl(signups_raw)`** Keep only the digits of each phone number (`digits`) and its last 9 digits (`phone9`). Show signup_id, digits and phone9, sorted by signup_id.

Expected output:

```
+---------+--------------+---------+
|signup_id|        digits|   phone9|
+---------+--------------+---------+
|        1|  998901234567|901234567|
|        2|00491512345678|512345678|
|        3|   34612345678|612345678|
|        4|     907654321|907654321|
|        5|   07700900123|700900123|
|        6|  998901234567|901234567|
+---------+--------------+---------+
```

In [87]:
signups_raw.show()

+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|signup_id|        full_name|             email|            phone|signup_date|amount|               prefs|
+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|        1|  aziz   KARIMOV |     Aziz@Mail.UZ |+998 90 123-45-67| 2024-01-05| 12.50|{"newsletter": tr...|
|        2|    bella schmidt|      bella@web.de| 0049 151 2345678| 20/01/2024|   abc|{"newsletter": fa...|
|        3|     Carlos  Ruiz|                  | (34) 612 345 678| 2024/02/02|     7|      {"lang": "es"}|
|        4| DILNOZA rahimova|   dilnoza@mail.uz|     90-765-43-21| 14/02/2024|      |                NULL|
|        5|      Emma Wilson|emma[at]post.co.uk|     07700 900123| 2024-03-01|  3.75|{"newsletter": true}|
|        6|     aziz karimov|      aziz@mail.uz|    +998901234567| 2024-03-02|  1.00|{"newsletter": fa...|
+---------+-----------------+--------

In [98]:
def etl(signups_raw):
    signups_raw=signups_raw.select(
        'signup_id',
        F.regexp_replace('phone','\D','').alias('digits'),
        F.substring(F.regexp_replace('phone',r'\D','').alias('digits'),-9,9).alias('digits9')
    )
    return signups_raw

etl(signups_raw).show()

+---------+--------------+---------+
|signup_id|        digits|  digits9|
+---------+--------------+---------+
|        1|  998901234567|901234567|
|        2|00491512345678|512345678|
|        3|   34612345678|612345678|
|        4|     907654321|907654321|
|        5|   07700900123|700900123|
|        6|  998901234567|901234567|
+---------+--------------+---------+



<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
/var/folders/bs/566ykyn56zs75rbs669wdt3m0000gn/T/ipykernel_89260/2281652856.py:4: SyntaxWarning: invalid escape sequence '\D'
  F.regexp_replace('phone','\D','').alias('digits'),


**Q15 · `etl(reviews)`** Clean the comments: remove the spaces at both ends, make them lower case, and remove every character that is not a letter, a digit or a space. Show review_id and clean_comment, sorted by review_id.

Expected output:

```
+---------+-----------------------------+
|review_id|clean_comment                |
+---------+-----------------------------+
|1        |great laptop very fast       |
|2        |nice story                   |
|3        |sound is ok delivery was slow|
|4        |broke after one week         |
|5        |fast and powerful            |
|6        |too long boring              |
|7        |useful recipes               |
|8        |good coffee slow start       |
+---------+-----------------------------+
```

In [104]:
def etl(reviews):
    reviews=reviews.select(
        'review_id',
        F.lower(F.regexp_replace('comment',r'[^a-zA-Z0-9\s]','')).alias('clean_comment')
    )

    return reviews

etl(reviews).show(truncate=False)

+---------+-----------------------------+
|review_id|clean_comment                |
+---------+-----------------------------+
|1        |  great laptop very fast     |
|2        |nice story                   |
|3        |sound is ok delivery was slow|
|4        |broke after one week         |
|5        |fast and powerful            |
|6        |too long boring              |
|7        |useful recipes               |
|8        |good coffee slow start       |
+---------+-----------------------------+



**Q16 · `etl(signups_raw)`** Extract the language code from `prefs` (the two letters in `"lang": "uz"`) as `lang`, null when there is none. Show signup_id and lang, sorted by signup_id.

Expected output:

```
+---------+----+
|signup_id|lang|
+---------+----+
|        1|  uz|
|        2|  de|
|        3|  es|
|        4|NULL|
|        5|NULL|
|        6|  uz|
+---------+----+
```

In [105]:
signups_raw.show()

+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|signup_id|        full_name|             email|            phone|signup_date|amount|               prefs|
+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|        1|  aziz   KARIMOV |     Aziz@Mail.UZ |+998 90 123-45-67| 2024-01-05| 12.50|{"newsletter": tr...|
|        2|    bella schmidt|      bella@web.de| 0049 151 2345678| 20/01/2024|   abc|{"newsletter": fa...|
|        3|     Carlos  Ruiz|                  | (34) 612 345 678| 2024/02/02|     7|      {"lang": "es"}|
|        4| DILNOZA rahimova|   dilnoza@mail.uz|     90-765-43-21| 14/02/2024|      |                NULL|
|        5|      Emma Wilson|emma[at]post.co.uk|     07700 900123| 2024-03-01|  3.75|{"newsletter": true}|
|        6|     aziz karimov|      aziz@mail.uz|    +998901234567| 2024-03-02|  1.00|{"newsletter": fa...|
+---------+-----------------+--------

In [110]:
def etl(signups_raw):
    signups_raw=signups_raw.select(
        'signup_id',
        F.regexp_extract('prefs',r'"lang"\s*:\s*"([a-zA-Z]{2})"',1).alias('lang')
    )

    return signups_raw.replace('',None)

etl(signups_raw).show()

+---------+----+
|signup_id|lang|
+---------+----+
|        1|  uz|
|        2|  de|
|        3|  es|
|        4|NULL|
|        5|NULL|
|        6|  uz|
+---------+----+



**Q17 · `etl(signups_raw)`** Keep the sign-ups whose amount is a plain number: digits, optionally followed by a dot and more digits (`12.50`, `7`). Show signup_id and `amount_value` (the amount as a double), sorted by signup_id.

Expected output:

```
+---------+------------+
|signup_id|amount_value|
+---------+------------+
|        1|        12.5|
|        3|         7.0|
|        5|        3.75|
|        6|         1.0|
+---------+------------+
```

In [112]:
signups_raw.show()

+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|signup_id|        full_name|             email|            phone|signup_date|amount|               prefs|
+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|        1|  aziz   KARIMOV |     Aziz@Mail.UZ |+998 90 123-45-67| 2024-01-05| 12.50|{"newsletter": tr...|
|        2|    bella schmidt|      bella@web.de| 0049 151 2345678| 20/01/2024|   abc|{"newsletter": fa...|
|        3|     Carlos  Ruiz|                  | (34) 612 345 678| 2024/02/02|     7|      {"lang": "es"}|
|        4| DILNOZA rahimova|   dilnoza@mail.uz|     90-765-43-21| 14/02/2024|      |                NULL|
|        5|      Emma Wilson|emma[at]post.co.uk|     07700 900123| 2024-03-01|  3.75|{"newsletter": true}|
|        6|     aziz karimov|      aziz@mail.uz|    +998901234567| 2024-03-02|  1.00|{"newsletter": fa...|
+---------+-----------------+--------

In [118]:
def etl(signups_raw):
    signups_raw=signups_raw.select(
        'signup_id',
        F.when(
        F.col('amount').rlike(r'^-?\d+(\.\d+)?$').alias('amount_value'),
        F.col('amount').cast('double')
        ).alias('amount_value')
                )
    return signups_raw.dropna()

etl(signups_raw).show()

+---------+------------+
|signup_id|amount_value|
+---------+------------+
|        1|        12.5|
|        3|         7.0|
|        5|        3.75|
|        6|         1.0|
+---------+------------+



## Lesson 7 · Duplicates, sorting and unions

**Duplicates**

| Method | What it keeps |
|---|---|
| `df.distinct()` | one copy of rows that are identical in **every** column |
| `df.dropDuplicates(["a", "b"])` | one row per (a, b) combination, but **which** row is not defined. When it matters ("keep the latest"), use `row_number` (Day 14). |
| `df.select("city").distinct()` | the different values of one column |

**Sorting.** `orderBy` (same as `sort`) takes several columns. For descending order, use `F.desc("x")` or `F.col("x").desc()`. Nulls come **first** in ascending order and **last** in descending order; change that with `F.col("x").asc_nulls_last()` or `.desc_nulls_first()`. `limit(n)` keeps the first n rows, so after `orderBy` it gives the top n. Zillacode ignores the row order, but a sorted result is much easier for you to check.

**Unions** stack the rows of two DataFrames:

| Method | How columns are matched |
|---|---|
| `a.union(b)`, `a.unionAll(b)` (same thing) | **by position**; both need the same number of columns. Keeps duplicates, like SQL `UNION ALL`. |
| `a.unionByName(b)` | **by name**, which is safer. `allowMissingColumns=True` fills a column that one side lacks with null. |
| `.distinct()` after the union | removes duplicates, like SQL `UNION` |

Before a union, make both sides look the same: `select` the same columns, use `withColumnRenamed` where the names differ, and add `F.lit("...")` as a tag when you need to know which table a row came from.

On Zillacode: #12 stacks two tables, #34 stacks three tables whose time columns have different names, #45 unions and then computes, and #18 and #24 remove duplicates.

In [119]:
print("events:", events.count(), "| distinct rows:", events.distinct().count(),
      "| one row per (customer, page):", events.dropDuplicates(["customer_id", "page"]).count())

events: 16 | distinct rows: 14 | one row per (customer, page): 8


In [120]:
customers.select("city").distinct().orderBy(F.col("city").asc_nulls_last()).show()

+--------+
|    city|
+--------+
|  Berlin|
|  London|
|  Madrid|
|Tashkent|
|    NULL|
+--------+



In [121]:
# union matches by position: with the columns in a different order, cities land in the name column
old = customers.filter(F.col("customer_id") <= 2).select("name", "city")
new = customers_new.filter(F.col("customer_id") <= 2).select("city", "name")
old.union(new).show()
old.unionByName(new).show()

+-------------+-------------+
|         name|         city|
+-------------+-------------+
| Aziz Karimov|     Tashkent|
|Bella Schmidt|       Berlin|
|     Tashkent| Aziz Karimov|
|       Munich|Bella Schmidt|
+-------------+-------------+

+-------------+--------+
|         name|    city|
+-------------+--------+
| Aziz Karimov|Tashkent|
|Bella Schmidt|  Berlin|
| Aziz Karimov|Tashkent|
|Bella Schmidt|  Munich|
+-------------+--------+



In [122]:
# Tag the source of each row, then stack
(customers.select("customer_id", "name", F.lit("old").alias("source"))
          .unionByName(customers_new.select("customer_id", "name", F.lit("new").alias("source")))
          .filter(F.col("customer_id") >= 6)
          .orderBy("customer_id", "source")
          .show())

+-----------+------------+------+
|customer_id|        name|source|
+-----------+------------+------+
|          6|Farid Aliyev|   new|
|          6|Farid Aliyev|   old|
|          7|   Grace Lee|   old|
|          8|   Hana Sato|   new|
+-----------+------------+------+



### Exercises

**Q18 · `etl(customers)`** Which cities do the customers live in? One row per city, no empty values, from A to Z.

Expected output:

```
+--------+
|    city|
+--------+
|  Berlin|
|  London|
|  Madrid|
|Tashkent|
+--------+
```

In [123]:
customers.show()

+-----------+----------------+----------------+--------+-----------+
|customer_id|            name|           email|    city|signup_date|
+-----------+----------------+----------------+--------+-----------+
|          1|    Aziz Karimov|    aziz@mail.uz|Tashkent| 2024-01-05|
|          2|   Bella Schmidt|    bella@web.de|  Berlin| 2024-01-20|
|          3|     Carlos Ruiz|carlos@correo.es|  Madrid| 2024-02-02|
|          4|Dilnoza Rahimova| dilnoza@mail.uz|Tashkent| 2024-02-14|
|          5|     Emma Wilson| emma@post.co.uk|  London| 2024-03-01|
|          6|    Farid Aliyev|            NULL|    NULL| 2024-03-10|
|          7|       Grace Lee|    grace@web.de|  Berlin| 2024-03-15|
+-----------+----------------+----------------+--------+-----------+



In [132]:
def etl(customers):
    customers=customers.select('city').distinct()
    return customers.orderBy('city').dropna()

etl(customers).show()

+--------+
|    city|
+--------+
|  Berlin|
|  London|
|  Madrid|
|Tashkent|
+--------+



**Q19 · `etl(customers, customers_new)`** List every customer who is in `customers` or in `customers_new`, once: customer_id and name, sorted by customer_id.

Expected output:

```
+-----------+----------------+
|customer_id|            name|
+-----------+----------------+
|          1|    Aziz Karimov|
|          2|   Bella Schmidt|
|          3|     Carlos Ruiz|
|          4|Dilnoza Rahimova|
|          5|     Emma Wilson|
|          6|    Farid Aliyev|
|          7|       Grace Lee|
|          8|       Hana Sato|
+-----------+----------------+
```

In [134]:
customers_new.show()
customers.show()

+-----------+----------------+----------------+--------+-----------+
|customer_id|            name|           email|    city|signup_date|
+-----------+----------------+----------------+--------+-----------+
|          1|    Aziz Karimov|    aziz@mail.uz|Tashkent| 2024-01-05|
|          2|   Bella Schmidt|    bella@web.de|  Munich| 2024-01-20|
|          3|     Carlos Ruiz|carlos@correo.es|  Madrid| 2024-02-02|
|          4|Dilnoza Rahimova| dilnoza@mail.uz|Tashkent| 2024-02-14|
|          5|     Emma Wilson| emma@post.co.uk|  London| 2024-03-01|
|          6|    Farid Aliyev|   farid@mail.az|    NULL| 2024-03-10|
|          8|       Hana Sato|    hana@mail.jp|   Osaka| 2024-04-02|
+-----------+----------------+----------------+--------+-----------+

+-----------+----------------+----------------+--------+-----------+
|customer_id|            name|           email|    city|signup_date|
+-----------+----------------+----------------+--------+-----------+
|          1|    Aziz Karimov|   

In [135]:
def etl(customers, customers_new):
    return (customers.select('customer_id','name')
            .unionByName(customers_new.select('customer_id','name'))
            .distinct()
        )

etl(customers, customers_new).show()

+-----------+----------------+
|customer_id|            name|
+-----------+----------------+
|          1|    Aziz Karimov|
|          2|   Bella Schmidt|
|          3|     Carlos Ruiz|
|          4|Dilnoza Rahimova|
|          5|     Emma Wilson|
|          6|    Farid Aliyev|
|          7|       Grace Lee|
|          8|       Hana Sato|
+-----------+----------------+



**Q20 · `etl(events)`** Show the 3 earliest events of the whole table, with duplicate rows removed: customer_id, event_time and page.

Expected output:

```
+-----------+-------------------+-------+
|customer_id|         event_time|   page|
+-----------+-------------------+-------+
|          1|2024-03-01 09:00:00|   home|
|          1|2024-03-01 09:10:00|product|
|          1|2024-03-01 11:00:00|   home|
+-----------+-------------------+-------+
```

In [136]:
events.show()

+-----------+-------------------+-------+
|customer_id|         event_time|   page|
+-----------+-------------------+-------+
|          1|2024-03-01 09:00:00|   home|
|          1|2024-03-01 09:10:00|product|
|          1|2024-03-01 09:10:00|product|
|          1|2024-03-01 11:00:00|   home|
|          1|2024-03-02 10:00:00|   cart|
|          1|2024-03-03 10:30:00|   home|
|          1|2024-03-05 08:00:00|   home|
|          2|2024-03-01 20:00:00|   home|
|          2|2024-03-01 20:20:00|product|
|          2|2024-03-01 20:45:00|   cart|
|          2|2024-03-04 18:00:00|   home|
|          2|2024-03-05 18:00:00|   home|
|          2|2024-03-05 18:00:00|   home|
|          3|2024-03-02 12:00:00|product|
|          3|2024-03-03 12:00:00|product|
|          3|2024-03-04 12:00:00|   cart|
+-----------+-------------------+-------+



In [137]:
def etl(events):
    events.select('customer_id','event_time','page').distinct()
    return events.limit(3)

etl(events).show()

+-----------+-------------------+-------+
|customer_id|         event_time|   page|
+-----------+-------------------+-------+
|          1|2024-03-01 09:00:00|   home|
|          1|2024-03-01 09:10:00|product|
|          1|2024-03-01 09:10:00|product|
+-----------+-------------------+-------+



## Mixed problems
Like Zillacode: several lessons in one problem. Plan the steps before you write code. About 10 to 15 minutes each.

**Q21 · `etl(signups_raw)`** Clean the sign-ups. `full_name`: no spaces at the ends, single spaces inside, each word capitalized. `email`: no spaces at the ends, lower case, null if it's empty or has no `@`. `amount`: a double, null if it isn't a number. Show signup_id, full_name, email and amount, sorted by signup_id.

Expected output:

```
+---------+----------------+---------------+------+
|signup_id|       full_name|          email|amount|
+---------+----------------+---------------+------+
|        1|    Aziz Karimov|   aziz@mail.uz|  12.5|
|        2|   Bella Schmidt|   bella@web.de|  NULL|
|        3|     Carlos Ruiz|           NULL|   7.0|
|        4|Dilnoza Rahimova|dilnoza@mail.uz|  NULL|
|        5|     Emma Wilson|           NULL|  3.75|
|        6|    Aziz Karimov|   aziz@mail.uz|   1.0|
+---------+----------------+---------------+------+
```

In [139]:
signups_raw.show()

+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|signup_id|        full_name|             email|            phone|signup_date|amount|               prefs|
+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|        1|  aziz   KARIMOV |     Aziz@Mail.UZ |+998 90 123-45-67| 2024-01-05| 12.50|{"newsletter": tr...|
|        2|    bella schmidt|      bella@web.de| 0049 151 2345678| 20/01/2024|   abc|{"newsletter": fa...|
|        3|     Carlos  Ruiz|                  | (34) 612 345 678| 2024/02/02|     7|      {"lang": "es"}|
|        4| DILNOZA rahimova|   dilnoza@mail.uz|     90-765-43-21| 14/02/2024|      |                NULL|
|        5|      Emma Wilson|emma[at]post.co.uk|     07700 900123| 2024-03-01|  3.75|{"newsletter": true}|
|        6|     aziz karimov|      aziz@mail.uz|    +998901234567| 2024-03-02|  1.00|{"newsletter": fa...|
+---------+-----------------+--------

In [150]:
def etl(signups_raw):
    signups_raw=signups_raw.select(
        'signup_id',
        F.initcap(F.regexp_replace('full_name',r'\s+', ' ')).alias('full_name'),
        F.lower(F.regexp_replace('email',r'^\s+|\s+$','')).alias('email'),
        F.col('amount').cast('double').alias('amount')
    )

    return signups_raw.replace('',None).orderBy('signup_id',ascending=False)

etl(signups_raw).show()

+---------+----------------+------------------+------+
|signup_id|       full_name|             email|amount|
+---------+----------------+------------------+------+
|        6|    Aziz Karimov|      aziz@mail.uz|   1.0|
|        5|     Emma Wilson|emma[at]post.co.uk|  3.75|
|        4|Dilnoza Rahimova|   dilnoza@mail.uz|  NULL|
|        3|     Carlos Ruiz|              NULL|   7.0|
|        2|   Bella Schmidt|      bella@web.de|  NULL|
|        1|   Aziz Karimov |      aziz@mail.uz|  12.5|
+---------+----------------+------------------+------+



**Q22 · `etl(reviews)`** Show review_id, rating, `sentiment` (`positive` for 4 or 5, `neutral` for 3, `negative` for 1 or 2) and `delivery_issue` (true when the tags contain `slow delivery`, false otherwise, also when there are no tags), sorted by review_id.

Expected output:

```
+---------+------+---------+--------------+
|review_id|rating|sentiment|delivery_issue|
+---------+------+---------+--------------+
|        1|     5| positive|         false|
|        2|     4| positive|         false|
|        3|     3|  neutral|          true|
|        4|     1| negative|          true|
|        5|     5| positive|         false|
|        6|     2| negative|         false|
|        7|     4| positive|         false|
|        8|     4| positive|         false|
+---------+------+---------+--------------+
```

In [161]:
def etl(reviews):
    reviews = reviews.select(
        'review_id',
        'rating',
        F.when(
            F.col('rating')>=4,"positive"
        ).when(
            F.col('rating')==3,'neutral'            
        ).when(
            F.col('rating')<=2,'negative'
        ).alias('sentiment'),

        F.coalesce(
            F.col('tags').contains('slow delivery'),
            F.lit(False)
        ).alias('delievry_problem')
    )
    return reviews.fillna(False)

etl(reviews).show()

+---------+------+---------+----------------+
|review_id|rating|sentiment|delievry_problem|
+---------+------+---------+----------------+
|        1|     5| positive|           false|
|        2|     4| positive|           false|
|        3|     3|  neutral|            true|
|        4|     1| negative|            true|
|        5|     5| positive|           false|
|        6|     2| negative|           false|
|        7|     4| positive|           false|
|        8|     4| positive|           false|
+---------+------+---------+----------------+



**Q23 · `etl(customers, signups_raw)`** List the different email domains (the part after `@`, lower case, without spaces) found in `customers` and in `signups_raw`. Skip missing emails and emails without `@`. One row per domain, from A to Z.

Expected output:

```
+----------+
|    domain|
+----------+
| correo.es|
|   mail.uz|
|post.co.uk|
|    web.de|
+----------+
```

In [162]:
customers.show()

+-----------+----------------+----------------+--------+-----------+
|customer_id|            name|           email|    city|signup_date|
+-----------+----------------+----------------+--------+-----------+
|          1|    Aziz Karimov|    aziz@mail.uz|Tashkent| 2024-01-05|
|          2|   Bella Schmidt|    bella@web.de|  Berlin| 2024-01-20|
|          3|     Carlos Ruiz|carlos@correo.es|  Madrid| 2024-02-02|
|          4|Dilnoza Rahimova| dilnoza@mail.uz|Tashkent| 2024-02-14|
|          5|     Emma Wilson| emma@post.co.uk|  London| 2024-03-01|
|          6|    Farid Aliyev|            NULL|    NULL| 2024-03-10|
|          7|       Grace Lee|    grace@web.de|  Berlin| 2024-03-15|
+-----------+----------------+----------------+--------+-----------+



In [163]:
signups_raw.show()

+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|signup_id|        full_name|             email|            phone|signup_date|amount|               prefs|
+---------+-----------------+------------------+-----------------+-----------+------+--------------------+
|        1|  aziz   KARIMOV |     Aziz@Mail.UZ |+998 90 123-45-67| 2024-01-05| 12.50|{"newsletter": tr...|
|        2|    bella schmidt|      bella@web.de| 0049 151 2345678| 20/01/2024|   abc|{"newsletter": fa...|
|        3|     Carlos  Ruiz|                  | (34) 612 345 678| 2024/02/02|     7|      {"lang": "es"}|
|        4| DILNOZA rahimova|   dilnoza@mail.uz|     90-765-43-21| 14/02/2024|      |                NULL|
|        5|      Emma Wilson|emma[at]post.co.uk|     07700 900123| 2024-03-01|  3.75|{"newsletter": true}|
|        6|     aziz karimov|      aziz@mail.uz|    +998901234567| 2024-03-02|  1.00|{"newsletter": fa...|
+---------+-----------------+--------

In [172]:
def etl(customers, signups_raw):
    emails=(customers.select(F.col('email'))
            .unionByName(signups_raw.select('email'))
                )
    return (
        emails.filter(F.col('email').contains('@'))
        .select(F.substring_index('email','@',-1).alias('domain_name'))
        .distinct()
        .orderBy('domain_name')
            )

etl(customers, signups_raw).show()

+-----------+
|domain_name|
+-----------+
|   Mail.UZ |
|  correo.es|
|    mail.uz|
| post.co.uk|
|     web.de|
+-----------+



## Answers
Try each exercise yourself first. Click an exercise to open its answer.

<details>
<summary><b>Q1</b> &nbsp;Show product_name, price and price_eur (the price in euros: 1 dollar = 0.92 euro, rounded to 2 decimals), ...</summary>

```python
def etl(products):
    return (products.select("product_name", "price", F.round(F.col("price") * 0.92, 2).alias("price_eur"))
                    .orderBy(F.desc("price")))

etl(products).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;Return the products with the same columns, but with the price as a whole number (int) and the product_name in ...</summary>

```python
def etl(products):
    return (products.withColumn("price", F.col("price").cast("int"))
                    .withColumn("product_name", F.upper("product_name"))
                    .orderBy("product_id"))

etl(products).show()
```

`withColumn` with an existing name replaces the column in its place, so the schema keeps its column names.

</details>

<details>
<summary><b>Q3</b> &nbsp;Show the orders that are shipped or cancelled and have a quantity of 2 or more: order_id, status and ...</summary>

```python
def etl(orders):
    return (orders.filter(F.col("status").isin("shipped", "cancelled") & (F.col("quantity") >= 2))
                  .select("order_id", "status", "quantity")
                  .orderBy("order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q4</b> &nbsp;Show the customers whose email is from Uzbekistan or Germany (it ends with .uz or .de): name and email, ...</summary>

```python
def etl(customers):
    return (customers.filter(F.col("email").endswith(".uz") | F.col("email").endswith(".de"))
                     .select("name", "email")
                     .orderBy("name"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q5</b> &nbsp;Show every customer who doesn't live in Berlin, customers without a city included: customer_id, name and ...</summary>

```python
def etl(customers):
    return (customers.filter((F.col("city") != "Berlin") | F.col("city").isNull())
                     .select("customer_id", "name", "city")
                     .orderBy("customer_id"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q6</b> &nbsp;Give every product a price_level: cheap if the price is under 50, mid from 50 to 200, premium above 200. Show ...</summary>

```python
def etl(products):
    return (products.withColumn("price_level",
                                F.when(F.col("price") < 50, "cheap")
                                 .when(F.col("price") <= 200, "mid")
                                 .otherwise("premium"))
                    .select("product_name", "price", "price_level")
                    .orderBy("price"))

etl(products).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;Label every order: problem if it was cancelled, otherwise guest if it has no customer_id, otherwise big if ...</summary>

```python
def etl(orders):
    return (orders.select("order_id",
                          F.when(F.col("status") == "cancelled", "problem")
                           .when(F.col("customer_id").isNull(), "guest")
                           .when(F.col("quantity") >= 3, "big")
                           .otherwise("normal").alias("label"))
                  .orderBy("order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q8</b> &nbsp;Show customer_id, name and city of every customer, with Unknown where the city is missing. Call the name ...</summary>

```python
def etl(customers):
    return (customers.withColumnRenamed("name", "customer_name")
                     .withColumn("city", F.coalesce(F.col("city"), F.lit("Unknown")))
                     .select("customer_id", "customer_name", "city")
                     .orderBy("customer_id"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q9</b> &nbsp;An empty email ("") means "no email". Show signup_id, email (null instead of an empty text) and has_email ...</summary>

```python
def etl(signups_raw):
    email = F.when(F.col("email") != "", F.col("email"))
    return (signups_raw.select("signup_id", email.alias("email"), email.isNotNull().alias("has_email"))
                       .orderBy("signup_id"))

etl(signups_raw).show()
```

</details>

<details>
<summary><b>Q10</b> &nbsp;Split name into first_name and last_name. Show customer_id, first_name and last_name, sorted by customer_id.</summary>

```python
def etl(customers):
    parts = F.split(F.col("name"), " ")
    return (customers.select("customer_id", parts[0].alias("first_name"), parts[1].alias("last_name"))
                     .orderBy("customer_id"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q11</b> &nbsp;Make a short name for every customer: the first name, a space, the first letter of the last name and a dot, ...</summary>

```python
def etl(customers):
    parts = F.split(F.col("name"), " ")
    short_name = F.concat(parts[0], F.lit(" "), F.substring(parts[1], 1, 1), F.lit("."))
    return customers.select("customer_id", short_name.alias("short_name")).orderBy("customer_id")

etl(customers).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;Mask the emails: keep the first letter, then *, then everything from the @ on (like a*@mail.uz). Customers ...</summary>

```python
def etl(customers):
    return (customers.filter(F.col("email").isNotNull())
                     .select("name",
                             F.concat(F.substring("email", 1, 1), F.lit("***@"), F.substring_index("email", "@", -1)).alias("masked_email"))
                     .orderBy("name"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;Give every product a code: the first 3 letters of its category in upper case, a dash, and the product_id, ...</summary>

```python
def etl(products):
    code = F.concat(F.upper(F.substring("category", 1, 3)), F.lit("-"), F.col("product_id").cast("string"))
    return products.select("product_name", code.alias("code")).orderBy("code")

etl(products).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;Keep only the digits of each phone number (digits) and its last 9 digits (phone9). Show signup_id, digits and ...</summary>

```python
def etl(signups_raw):
    return (signups_raw.withColumn("digits", F.regexp_replace("phone", "[^0-9]", ""))
                       .select("signup_id", "digits", F.substring("digits", -9, 9).alias("phone9"))
                       .orderBy("signup_id"))

etl(signups_raw).show()
```

</details>

<details>
<summary><b>Q15</b> &nbsp;Clean the comments: remove the spaces at both ends, make them lower case, and remove every character that is ...</summary>

```python
def etl(reviews):
    return (reviews.select("review_id",
                            F.regexp_replace(F.lower(F.trim("comment")), "[^a-z0-9 ]", "").alias("clean_comment"))
                   .orderBy("review_id"))

etl(reviews).show(truncate=False)
```

</details>

<details>
<summary><b>Q16</b> &nbsp;Extract the language code from prefs (the two letters in "lang": "uz") as lang, null when there is none. Show ...</summary>

```python
def etl(signups_raw):
    lang = F.regexp_extract("prefs", r'"lang": "([a-z]{2})"', 1)
    return signups_raw.select("signup_id", F.when(lang != "", lang).alias("lang")).orderBy("signup_id")

etl(signups_raw).show()
```

Row 4 has no prefs at all: `regexp_extract` of null is null. Row 5 has prefs without a lang: that gives `""`, which `when` turns into null.

</details>

<details>
<summary><b>Q17</b> &nbsp;Keep the sign-ups whose amount is a plain number: digits, optionally followed by a dot and more digits ...</summary>

```python
def etl(signups_raw):
    return (signups_raw.filter(F.col("amount").rlike(r"^\d+(\.\d+)?$"))
                       .select("signup_id", F.col("amount").cast("double").alias("amount_value"))
                       .orderBy("signup_id"))

etl(signups_raw).show()
```

</details>

<details>
<summary><b>Q18</b> &nbsp;Which cities do the customers live in? One row per city, no empty values, from A to Z.</summary>

```python
def etl(customers):
    return customers.select("city").dropna().distinct().orderBy("city")

etl(customers).show()
```

</details>

<details>
<summary><b>Q19</b> &nbsp;List every customer who is in customers or in customers_new, once: customer_id and name, sorted by ...</summary>

```python
def etl(customers, customers_new):
    return (customers.select("customer_id", "name")
                     .unionByName(customers_new.select("customer_id", "name"))
                     .distinct()
                     .orderBy("customer_id"))

etl(customers, customers_new).show()
```

</details>

<details>
<summary><b>Q20</b> &nbsp;Show the 3 earliest events of the whole table, with duplicate rows removed: customer_id, event_time and page.</summary>

```python
def etl(events):
    return events.distinct().orderBy("event_time").limit(3)

etl(events).show()
```

</details>

<details>
<summary><b>Q21</b> &nbsp;Clean the sign-ups. full_name: no spaces at the ends, single spaces inside, each word capitalized. email: no ...</summary>

```python
def etl(signups_raw):
    email = F.lower(F.trim("email"))
    return (signups_raw.select(
                "signup_id",
                F.initcap(F.trim(F.regexp_replace("full_name", r"\s+", " "))).alias("full_name"),
                F.when(email.contains("@"), email).alias("email"),
                F.col("amount").cast("double").alias("amount"))
            .orderBy("signup_id"))

etl(signups_raw).show()
```

With ANSI mode off (Zillacode, and this notebook), `cast` turns `abc` and `""` into null. In Spark 4 with ANSI on you would write `F.col("amount").try_cast("double")`.

</details>

<details>
<summary><b>Q22</b> &nbsp;Show review_id, rating, sentiment (positive for 4 or 5, neutral for 3, negative for 1 or 2) and ...</summary>

```python
def etl(reviews):
    return (reviews.select(
                "review_id", "rating",
                F.when(F.col("rating") >= 4, "positive").when(F.col("rating") == 3, "neutral").otherwise("negative").alias("sentiment"),
                F.coalesce(F.col("tags").contains("slow delivery"), F.lit(False)).alias("delivery_issue"))
            .orderBy("review_id"))

etl(reviews).show()
```

`contains` on a null text is null, not false, so `coalesce(..., False)` is needed for review 6.

</details>

<details>
<summary><b>Q23</b> &nbsp;List the different email domains (the part after @, lower case, without spaces) found in customers and in ...</summary>

```python
def etl(customers, signups_raw):
    emails = (customers.select("email")
                       .unionByName(signups_raw.select("email"))
                       .select(F.lower(F.trim("email")).alias("email")))
    return (emails.filter(F.col("email").contains("@"))
                  .select(F.substring_index("email", "@", -1).alias("domain"))
                  .distinct()
                  .orderBy("domain"))

etl(customers, signups_raw).show()
```

</details>